# MicroPlant — Model Training

This notebook trains the MicroPlant model for plant disease classification.

We explore:
- Baseline training (MicroPlant)
- Teacher model (ResNet18)
- Knowledge Distillation (KD)

The goal is to build an accurate yet lightweight model suitable for edge deployment.

In [15]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.preprocessing import get_dataloaders
from src.architectures import get_microplant, get_teacher_model
from src.training import train_model, validate
from src.utils import count_parameters, count_model_bytes

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="torch.utils.data.dataloader")

## Configuration

We define training parameters and device setup.

In [16]:
DATA_DIR = "../data/MicroPlantDataset"

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

BATCH_SIZE = 64
EPOCHS = 10
LR = 0.001

print(f"using : {DEVICE}")

using : cpu


## Load Dataset

We use the predefined data pipeline with augmentation and proper splitting.

In [17]:
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    DATA_DIR, batch_size=BATCH_SIZE
)

print("Classes:", class_names)

Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy']


## Baseline Model (MicroPlant)

We first train the MicroPlant model without any distillation.

This serves as a baseline for comparison.

In [18]:
baseline_model = get_microplant(num_classes=len(class_names)).to(DEVICE)

print("Parameters:", count_parameters(baseline_model))
count_model_bytes(baseline_model)

Parameters: 8804
Model Weights: 35,216 bytes
Model Buffers: 2,648 bytes
Total Size: 36.98 KB


37864

In [19]:
baseline_model = train_model(
    baseline_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    save_name="../models/microplant_baseline",
    device=DEVICE
)

Training: 100%|██████████| 41/41 [00:25<00:00,  1.60it/s]


Epoch 1 | Train Loss 1.2183 Acc 0.46 | Val Loss 1.2480 F1 0.4101


Training: 100%|██████████| 41/41 [00:21<00:00,  1.93it/s]


Epoch 2 | Train Loss 0.8509 Acc 0.64 | Val Loss 0.6523 F1 0.6775


Training: 100%|██████████| 41/41 [00:23<00:00,  1.76it/s]


...

Epoch 8 | Train Loss 0.2236 Acc 0.93 | Val Loss 0.1417 F1 0.9540


Training: 100%|██████████| 41/41 [00:20<00:00,  1.96it/s]


Epoch 9 | Train Loss 0.1941 Acc 0.93 | Val Loss 0.1351 F1 0.9538


Training: 100%|██████████| 41/41 [00:23<00:00,  1.72it/s]


Epoch 10 | Train Loss 0.2184 Acc 0.94 | Val Loss 0.1282 F1 0.9434


In [20]:
baseline_loss, baseline_f1 = validate(baseline_model, test_loader, device=DEVICE)

print("Baseline F1:", baseline_f1)

Baseline F1: 0.934358582994983


## Teacher Model (ResNet18)

We train a larger model to act as a teacher.

This model provides strong supervision for distillation.

In [5]:
teacher_model = get_teacher_model(num_classes=len(class_names)).to(DEVICE)

print("Teacher parameters:", count_parameters(teacher_model))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\hp/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:06<00:00, 6.90MB/s]


Teacher parameters: 11178564


In [6]:
teacher_model = train_model(
    teacher_model,
    train_loader,
    val_loader,
    epochs=10,
    lr=LR,
    save_name="../models/teacher",
    device=DEVICE
)

Training: 100%|██████████| 41/41 [04:54<00:00,  7.18s/it]


Epoch 1 | Train Loss 0.1548 Acc 0.94 | Val Loss 0.2078 F1 0.9241


Training: 100%|██████████| 41/41 [04:26<00:00,  6.50s/it]


Epoch 2 | Train Loss 0.0588 Acc 0.98 | Val Loss 0.0820 F1 0.9719


Training: 100%|██████████| 41/41 [04:25<00:00,  6.48s/it]


...

Epoch 8 | Train Loss 0.0235 Acc 0.99 | Val Loss 0.0035 F1 1.0000


Training: 100%|██████████| 41/41 [04:27<00:00,  6.53s/it]


Epoch 9 | Train Loss 0.0218 Acc 0.99 | Val Loss 0.0019 F1 1.0000


Training: 100%|██████████| 41/41 [04:23<00:00,  6.42s/it]


Epoch 10 | Train Loss 0.0278 Acc 1.00 | Val Loss 0.0030 F1 1.0000


In [7]:
teacher_loss, teacher_f1 = validate(teacher_model, test_loader, device=DEVICE)

print("Teacher F1:", teacher_f1)

Teacher F1: 1.0


## Knowledge Distillation

We train the MicroPlant model using guidance from the teacher model.

The student learns from:
- Ground truth labels
- Teacher predictions (soft targets)

In [8]:
student_model = get_microplant(num_classes=len(class_names)).to(DEVICE)

In [9]:
student_model = train_model(
    student_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    teacher=teacher_model,
    kd_alpha=0.7,
    kd_temp=3.0,
    lr=LR,
    save_name="../models/microplant_kd",
    device=DEVICE
)

Training: 100%|██████████| 41/41 [01:49<00:00,  2.66s/it]


Epoch 1 | Train Loss 5.9242 Acc 0.47 | Val Loss 1.2204 F1 0.3791


Training: 100%|██████████| 41/41 [01:51<00:00,  2.71s/it]


Epoch 2 | Train Loss 4.3702 Acc 0.67 | Val Loss 0.6149 F1 0.7807


Training: 100%|██████████| 41/41 [01:48<00:00,  2.65s/it]


...

Epoch 8 | Train Loss 0.9863 Acc 0.94 | Val Loss 0.2072 F1 0.9465


Training: 100%|██████████| 41/41 [01:49<00:00,  2.67s/it]


Epoch 9 | Train Loss 0.8410 Acc 0.95 | Val Loss 0.1705 F1 0.9532


Training: 100%|██████████| 41/41 [01:48<00:00,  2.64s/it]


Epoch 10 | Train Loss 0.8328 Acc 0.95 | Val Loss 0.1769 F1 0.9571


In [10]:
student_loss, student_f1 = validate(student_model, test_loader, device=DEVICE)

print("Student (KD) F1:", student_f1)

Student (KD) F1: 0.9525589791964324


## Model Comparison

We compare the performance of:
- Baseline MicroPlant
- Teacher (ResNet18)
- Distilled MicroPlant

In [21]:
print("=== Final Results ===")
print(f"Baseline F1: {baseline_f1:.4f}")
print(f"Teacher F1:  {teacher_f1:.4f}")
print(f"Student F1:  {student_f1:.4f}")

=== Final Results ===
Baseline F1: 0.9344
Teacher F1:  1.0000
Student F1:  0.9526


## Observations

- The teacher model achieves the highest performance due to its capacity
- The distilled student improves over the baseline model
- Knowledge distillation helps transfer knowledge into a smaller model

### Conclusion
The MicroPlant model benefits from distillation, achieving strong performance while remaining lightweight and suitable for edge deployment.